In [ ]:
import glob, os, shutil, subprocess, sys
src = os.path.dirname(glob.glob("/kaggle/input/**/train_encoder.py", recursive=True)[0])
print("bundle at", src)
shutil.copytree(src, "/kaggle/working/src", dirs_exist_ok=True)
pkg = "/kaggle/working/src/decide"
os.makedirs(pkg, exist_ok=True)
open(os.path.join(pkg, "__init__.py"), "w").close()
for m in ("features.py", "tabular.py", "inputs.py"):
    if os.path.exists(os.path.join("/kaggle/working/src", m)):
        shutil.move(os.path.join("/kaggle/working/src", m), os.path.join(pkg, m))
os.makedirs("/kaggle/working/src/data", exist_ok=True)
for f in ("train.jsonl", "dev.jsonl"):
    shutil.copy(glob.glob(f"/kaggle/input/**/{f}", recursive=True)[0], "/kaggle/working/src/data/" + f)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript"], check=True)
for name, base in [("h768", "cross-encoder/nli-MiniLM2-L6-H768"), ("h384", "cross-encoder/ms-marco-MiniLM-L6-v2")]:
    subprocess.run([sys.executable, "train_encoder.py", "--data", "data", "--out", f"/kaggle/working/art_{name}",
                    "--epochs", "5", "--base", base], cwd="/kaggle/working/src", check=True)
    shutil.make_archive(f"/kaggle/working/art_{name}", "zip", f"/kaggle/working/art_{name}")
    shutil.rmtree(f"/kaggle/working/art_{name}")
shutil.rmtree("/kaggle/working/src")
print(os.listdir("/kaggle/working"))
